# 6. XGBoost

SECOM 라벨을 PASS=0, FAIL=1로 변환해 XGBoost를 학습하고 불량 클래스 성능을 평가합니다.

각 노트북은 학습/테스트 분할부터 독립 실행할 수 있습니다. 모델 전처리는 학습 데이터에만 맞춰 테스트 데이터 누수를 방지합니다.

## 공통 데이터 준비

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

data_dir = Path.cwd() / "data"
if not data_dir.is_dir():
    data_dir = Path.cwd().parent / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")

X_raw = pd.read_csv(data_dir / "secom.data", sep=r"\s+", header=None)
labels = pd.read_csv(data_dir / "secom_labels.data", sep=r"\s+", header=None)
y = labels.iloc[:, 0].astype(int)

if len(X_raw) != len(y):
    raise ValueError(f"센서 데이터와 라벨 행 수가 다릅니다: {len(X_raw)} != {len(y)}")

# 분할 후 학습 데이터에서만 결측률 기준과 중앙값을 학습해 데이터 누수를 막습니다.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
keep_columns = X_train_raw.columns[X_train_raw.isna().mean() < 0.8]
X_train_raw = X_train_raw.loc[:, keep_columns]
X_test_raw = X_test_raw.loc[:, keep_columns]

imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=keep_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=keep_columns,
    index=X_test_raw.index,
)

keep_columns = X_train.columns[X_train.nunique(dropna=False) > 1]
X_train = X_train.loc[:, keep_columns]
X_test = X_test.loc[:, keep_columns]

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print("학습 라벨 분포:")
print(y_train.value_counts().sort_index())


학습 데이터: (1253, 466), 테스트 데이터: (314, 466)
학습 라벨 분포:
0
-1    1170
 1      83
Name: count, dtype: int64


## 라벨 변환 및 모델 학습

In [2]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

TARGET_NAMES = ["PASS (0)", "FAIL (1)"]


def report_binary_results(y_true, y_pred):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=TARGET_NAMES,
        zero_division=0,
    ))
    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


y_train_binary = (y_train == 1).astype(int)
y_test_binary = (y_test == 1).astype(int)

# 1. 불균형 비율 계산
scale_weight = (y_train_binary == 0).sum() / (y_train_binary == 1).sum()

# 2. 과적합 방지 및 소수 클래스 탐지를 위한 하이퍼파라미터 튜닝
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,                 # 트리 깊이를 낮춰 정상 데이터 과적합 방지
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    subsample=0.8,
    colsample_bytree=0.5,        # 센서 피처를 무작위 샘플링하여 다양한 센서 반영
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train_binary)

# 3. 불량 예측 확률 추출
pred_prob = xgb_model.predict_proba(X_test)[:, 1]

# 4. 상위 불량 후보군을 잡기 위한 최적 임계값(Threshold) 탐색 및 적용
# (실제 불량 평균 확률 수준인 0.1 전후를 기준으로 판정)
threshold = 0.3
pred_xgb = (pred_prob >= threshold).astype(int)

print(f"=== 적용된 임계값(Threshold): {threshold} ===")
report_binary_results(y_test_binary, pred_xgb)

=== 적용된 임계값(Threshold): 0.3 ===
Accuracy: 0.8439
              precision    recall  f1-score   support

    PASS (0)       0.97      0.86      0.91       293
    FAIL (1)       0.23      0.57      0.33        21

    accuracy                           0.84       314
   macro avg       0.60      0.72      0.62       314
weighted avg       0.92      0.84      0.87       314

Confusion matrix (actual rows, predicted columns):
[[253  40]
 [  9  12]]


In [3]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = data_dir.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def normalize_labels(values):
    values = pd.Series(values).reset_index(drop=True)
    unique_values = set(values.dropna().unique())

    if unique_values.issubset({-1, 1}):
        values = values.map({-1: 0, 1: 1})

    return values.astype(int)


MODEL_NAME = "06XGBoost1"
y_true_saved = normalize_labels(y_test)
y_pred_saved = normalize_labels(pred_xgb)

result_df = pd.DataFrame({
    "y_true": y_true_saved,
    "y_pred": y_pred_saved,
})

result_path = RESULTS_DIR / f"{MODEL_NAME}.csv"
result_df.to_csv(result_path, index=False)

print(f"저장 완료: {result_path.name}")
print(result_df.head())

저장 완료: 06XGBoost1.csv
   y_true  y_pred
0       0       0
1       0       0
2       0       0
3       0       0
4       0       0


In [4]:
import numpy as np

# 테스트 세트의 불량 확률 통계 확인
print("=== 테스트 데이터의 불량(1) 예측 확률 요약 ===")
print(f"최대 예측 확률: {pred_prob.max():.4f}")
print(f"평균 예측 확률: {pred_prob.mean():.4f}")
print(f"상위 10% 확률값: {np.percentile(pred_prob, 90):.4f}")
print(f"상위 5% 확률값:  {np.percentile(pred_prob, 95):.4f}")

# 실제 불량(21개) 샘플에 대해 모델이 준 예측 확률
actual_fail_probs = pred_prob[y_test_binary == 1]
print(f"\n실제 불량(FAIL) 21개의 평균 예측 확률: {actual_fail_probs.mean():.4f}")
print(f"실제 불량(FAIL) 중 가장 높은 예측 확률: {actual_fail_probs.max():.4f}")

=== 테스트 데이터의 불량(1) 예측 확률 요약 ===
최대 예측 확률: 0.7481
평균 예측 확률: 0.1760
상위 10% 확률값: 0.3584
상위 5% 확률값:  0.4255

실제 불량(FAIL) 21개의 평균 예측 확률: 0.3032
실제 불량(FAIL) 중 가장 높은 예측 확률: 0.5810


In [5]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
# 임계값을 0.1부터 0.6까지 0.05 단위로 테스트
for th in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    temp_pred = (pred_prob >= th).astype(int)
    
    # 실제 불량(21개) 중 맞춘 개수
    detected_fail = ((y_test_binary == 1) & (temp_pred == 1)).sum()
    # 정상(293개) 중 불량으로 오탐한 개수
    false_alarm = ((y_test_binary == 0) & (temp_pred == 1)).sum()
    
    results.append({
        "Threshold": th,
        "FAIL Recall": recall_score(y_test_binary, temp_pred, pos_label=1),
        "FAIL Precision": precision_score(y_test_binary, temp_pred, pos_label=1, zero_division=0),
        "FAIL F1": f1_score(y_test_binary, temp_pred, pos_label=1, zero_division=0),
        "검출된 불량(총 21개)": f"{detected_fail}개",
        "정상 오탐(총 293개)": f"{false_alarm}개"
    })

threshold_df = pd.DataFrame(results)
display(threshold_df)

,Threshold,FAIL Recall,FAIL Precision,FAIL F1,검출된 불량(총 21개),정상 오탐(총 293개)
0,0.10,0.904762,0.092683,0.168142,19개,186개
1,0.15,0.761905,0.104575,0.183908,16개,137개
2,0.20,0.666667,0.135922,0.225806,14개,89개
3,0.25,0.619048,0.194030,0.295455,13개,54개
4,0.30,0.571429,0.230769,0.328767,12개,40개
5,0.35,0.476190,0.270270,0.344828,10개,27개
6,0.40,0.333333,0.350000,0.341463,7개,13개
7,0.50,0.095238,0.285714,0.142857,2개,5개
